In [ ]:
import os
import json
from dotenv import load_dotenv
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import ElementMetadata
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_chroma import Chroma


load_dotenv()


def partition_pdf_document(file_path: str):
    elements = partition_pdf(filename=file_path, strategy="hi_res",
                             infer_table_structure=True,
                             extract_image_block_types=["Image"],
                             extract_image_block_to_payload=True)
    print(f"Partitioned {len(elements)} elements from the PDF document.")
    print("Sample elements:")
    print(elements[117].to_dict())
    return elements

elements = partition_pdf_document("/Users/joshil/personal/learning/RAG/docs/sample.pdf")

In [ ]:
len(elements)
for element in elements:
    e = element.to_dict()
    print(e["type"])
# for i, element in enumerate(elements):
#     print(f"Element {i}: {element.category}")

In [ ]:
def chunk_by_title_elements(elements):
    chunks = chunk_by_title(elements=elements,
                            max_characters=3000,
                            new_after_n_chars=2400,
                            combine_text_under_n_chars=500)

    return chunks

chunks = chunk_by_title_elements(elements)

In [ ]:
chunks[4].to_dict()
chunks[4].metadata.orig_elements[3].to_dict()

# chunks[4].to_dict()

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk.to_dict()['type']}")
    if hasattr(chunk.metadata, "orig_elements") and chunk.metadata.orig_elements:
        print(f"  Original elements in this chunk:")
        for orig_element in chunk.metadata.orig_elements:
            print(f"    - {orig_element.to_dict()['type']}")

In [ ]:



def get_chunk_contents(chunk):
    data = {
        'text' :chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for orig_element in chunk.metadata.orig_elements:
            
            element_type = orig_element.to_dict().get('type', 'unknown')
            
            if element_type == 'Table':
                html_table = getattr(orig_element.metadata, 'text_as_html', orig_element.text)
                data['tables'].append(html_table)
                data['types'].append('table')
            elif element_type == 'Image':
                if hasattr(orig_element, 'metadata') and hasattr(orig_element.metadata, 'image_base64'):
                    image = getattr(orig_element.metadata, 'image_base64', None)
                    if image:
                        data['images'].append(image)
                        data['types'].append('image')
            
    data['types'] = list(set(data['types']))
    print(f"Chunk  data:", data)
    return data


def summarize_chunks(chunks):
    langchain_document = []
    for chunk in chunks:
        content_data = get_chunk_contents(chunk)
         
        if content_data['table'] or content_data['images']:
            summarized_content = create_ai_enhanced_summary(content_data['text'], content_data['tables'], content_data['images'])
        else:
            summarized_content = content_data['text']

        doc = Document(
            page_content=summarized_content,
            metadata={
                "raw_text": content_data['text'],
                "tables_html": content_data['tables'],
                "images_base64": content_data['images'],
            }
        )
        langchain_document.append(doc)
    return langchain_document

content = get_chunk_contents(chunks[20])

In [ ]:
def create_ai_enhanced_summary(text:str, tables, images):
    llm = ChatOllama(model="qwen3.6")

    prompt = f"""You are an assistant that summarizes content for retrieval. Please create a concise summary of the following content. If there are tables, summarize the key insights from the tables. If there are images, describe what the images depict based on their captions or any available metadata.\n\n
                Content to analyze:\n
                Text Content:\n{text}\n\n
                """
    
    if tables:
        prompt += f"Tables content:\n"
        for i, table in enumerate(tables):
            prompt += f"Table {i+1}:\n{table}\n\n"

        prompt+= """
            Your task:
            
            Generate a concise summary and searchable description that covers:
            - Key insights from the text content.
            - Important information from the tables (e.g., trends, comparisons, key data points).
            - Descriptions of any images based on their captions or metadata.
            - Ensure the summary is informative and captures the essence of the content for effective retrieval.
            - Questions this content could answer.

            Make it detailed and searchable
        
        """
    print("Prompt for LLM:")
    print(prompt)
    
    message_content = [{"type": "text", "text": prompt}]

    for image in images:
        message_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image}"})

    message = HumanMessage(content=message_content)
    response = llm.invoke([message])
    print("LLM response:")  
    print(response)
    return response.content

print(content)
ai_summary = create_ai_enhanced_summary(content['text'], content['tables'], content['images'])
print(ai_summary)

In [20]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_chroma import Chroma
import json


def create_vector_store(chunks, persist_dir: str = "db/chroma_db_v2", model: str = "gemini-embedding-001"):
    match model.lower():
        case "gemini-embedding-001":
            embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
        case "qwen3-embedding":
            embedding_model = OllamaEmbeddings(model="qwen3-embedding")

        case _:
            raise ValueError(
                f"Unsupported embedding model: {model}. "
                "Supported values: gemini-embedding-001"
            )
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"Vector store created at: {persist_dir}")
    return vector_store


doc = Document(
            page_content=ai_summary,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content['text'],
                    "tables_html": content['tables'],
                    "images_base64": content['images'],
                })
            }
        )
print(doc)
print("--------------------------------")
db = create_vector_store([doc], persist_dir="db/chroma_db_v2", model="qwen3-embedding")


page_content='### **Content Summary & Searchable Description**
This content presents a hyperparameter ablation study and performance benchmark for a Transformer-based neural machine translation model. It systematically evaluates architectural variations—including model dimensionality (`dmodel`), layer count (`N`), dropout/regularization rates, and positional encoding schemes—using development-set Perplexity (PPL) and BLEU scores. The data highlights how scaling capacity, depth, and regularization directly impacts translation quality, establishing optimal training configurations for both base and large-scale models.

---

### **Key Insights from Text Content**
- **Two Primary Configurations:** A `base` model (512-dim, ~65M params) and a `big` model (1024-dim, ~213M params) are compared, with the `big` variant showing superior performance.
- **Ablation Series (A–E):** The text outlines systematic perturbations to isolate the impact of individual components:
  - Varying self-attention/out

In [21]:
retriever = db.as_retriever(search_kwargs={"k": 3})
query = "What are the parameters for a Transformer base model?"
chunks = retriever.invoke(query)

def generate_answer(query, chunks):


    llm = ChatOllama(model="qwen3.6")
    prompt = f"""You are an assistant that answers questions based on retrieved content chunks. Please provide a comprehensive answer to the following question using the information from the content. If the content contain tables, extract key insights from them. If there are images, describe what they depict based on any available metadata. Ensure your answer is detailed and directly addresses the question.
    Question: {query}  
    
    Content:"""

    for i, chunk in enumerate(chunks):
        prompt += f"\nDocument {i+1}:\n\n"
        if chunk.metadata.get("original_content"):
            original_content = json.loads(chunk.metadata["original_content"])
            
            raw_text = original_content.get("raw_text", "")
            if raw_text:
               prompt += f"Text Content:\n{raw_text}\n\n"
            
            tables = original_content.get("tables_html", [])
            
            if tables:
                prompt += "Tables \n"
                for j, table in enumerate(tables):
                    prompt += f"Table {j+1}:\n{table}\n"
            
            prompt += "\n\n"
    
    message_content = [{"type": "text", "text": prompt}]
    print("Prompt for LLM:")
    print(prompt)
    for chunk in chunks:
        if chunk.metadata.get("original_content"):
            original_content = json.loads(chunk.metadata["original_content"])
            images = original_content.get("images_base64", [])
            for image in images:
                message_content.append({"type": "image_url", "image_url": f"data:image/png;base64,{image}"})
    
    message = HumanMessage(content=message_content)
    response = llm.invoke([message])
    print("LLM response:")
    print(response.content)
    return response.content

answer = generate_answer(query, chunks)
print("Final Answer:")
print(answer)

<>:15: SyntaxWarning: invalid escape sequence '\D'
<>:15: SyntaxWarning: invalid escape sequence '\D'
/var/folders/29/c14rhw955n5dh3_schfwdmpr0000gq/T/ipykernel_93655/1985170833.py:15: SyntaxWarning: invalid escape sequence '\D'
  prompt += f"\n\Document {i+1}:\n\n"


Prompt for LLM:
You are an assistant that answers questions based on retrieved content chunks. Please provide a comprehensive answer to the following question using the information from the content. If the content contain tables, extract key insights from them. If there are images, describe what they depict based on any available metadata. Ensure your answer is detailed and directly addresses the question.
    Question: What are the parameters for a Transformer base model?  

    Content:
\Document 1:

Text Content:
train | PPL BLEU params N dmode er ho dy dy Farop 1s steps | (dev) (dev) x 108 base | 6 512 2048 8 64 64 0.1 0.1 100K | 4.92 25.8 65 1 512 512 5.29 24.9 (A) 4 128 128 5.00 25.5 16 32 32 4.91 25.8 32 =616 16 5.01 25.4 (B) 16 5.16 25.1 58 32 5.01 25.4 60 2 6.11 23.7 36 4 5.19 25.3 50 8 4.88 25.5 80 (C) 256 32 8632 5.75 24.5 28 1024 128 128 4.66 26.0 168 1024 5.12 25.4 53 4096 4.75 26.2 90 0.0 5.77 24.6 0.2 4.95 25.5 (D) 0.0 4.67 253 0.2 5.47 25.7 (E) positional embedding inst